This notebook is a standalone, minimal, and clean implementation used to reproduce the baseline results reported in the paper. 
It does not include the full research process (e.g., hyperparameter tuning or exploratory experiments). 
The goal is to provide a self-contained script that runs quickly and reproduces the baseline results presented in the main section of the paper.

In [15]:
import pandas as pd
import numpy as np
import itertools

from tslearn.datasets import UCR_UEA_datasets

import pycatch22

from sklearn.metrics import classification_report
import xgboost
from sklearn.preprocessing import LabelEncoder

from aeon.classification.convolution_based import MultiRocketClassifier

import warnings
warnings.filterwarnings("ignore")

SEED = 42

# Load Data

In [9]:
# Load the LSST dataset from UEA archive
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

In [10]:
def tweak_and_convert_y(y):
    return (
        pd.DataFrame(
            y,
            columns=["target"],
            index=pd.RangeIndex(len(y), name="Sample")
        )
        .astype("category")
    )
df_y_train = tweak_and_convert_y(y_train)
df_y_train

,target
Sample,
0,6
1,6
2,6
3,6
4,6
...,...
2454,95
2455,95
2456,95


In [11]:
def compute_catch22_features(X):
    n_samples, n_steps, n_channel = X.shape
    X_train_features = []
    for i in range(n_samples):
        features_sample = []
        for channel in range(n_channel):
            ts = X[i, :, channel]
            features_channel = pycatch22.catch22_all(ts)
            features_sample += features_channel["values"] 
        features_sample = np.array(features_sample)
        X_train_features.append(features_sample)

    X_train_features = np.array(X_train_features)
    X_train_features = (
        pd.DataFrame(
            X_train_features,
            columns=[f"{name}_{i}" for name, i in itertools.product(
                range(n_channel), 
                features_channel['names'])],
            index=pd.RangeIndex(n_samples, name="Sample")
        )
    )
    return X_train_features

X_train_features = compute_catch22_features(X_train)

In [12]:
X_test_features = compute_catch22_features(X_test)
X_test_features = X_test_features.fillna(0)
# only droping one.. 1295

In [16]:
def to_channels_first(X):
    """(n, T, C) -> (n, C, T)"""
    return np.transpose(X, (0, 2, 1))

X_train_cf = to_channels_first(X_train).astype(np.float32)
X_test_cf = to_channels_first(X_test).astype(np.float32)

# Model : Baseline 1 (Catch22 + XGBoost)

In [13]:
le = LabelEncoder()
encoded_target = le.fit_transform(df_y_train["target"])

XGB = xgboost.XGBClassifier(random_state=SEED, use_label_encoder=False, eval_metric="mlogloss")
XGB.fit(X_train_features, encoded_target)

y_pred_XGB = XGB.predict(X_test_features)

In [14]:
df_y_test = tweak_and_convert_y(y_test)
print("XGBoost:")
print(classification_report(
    df_y_test["target"], 
    le.inverse_transform(y_pred_XGB)
))

XGBoost:
              precision    recall  f1-score   support

          15       0.59      0.40      0.48       124
          16       0.80      0.87      0.84       270
          42       0.54      0.44      0.48       382
          52       0.00      0.00      0.00        63
          53       1.00      0.29      0.44         7
           6       0.70      0.20      0.31        35
          62       0.37      0.10      0.15       153
          64       0.00      0.00      0.00        24
          65       0.69      0.87      0.77       313
          67       0.20      0.01      0.03        68
          88       0.87      0.84      0.86       121
          90       0.57      0.84      0.68       777
          92       0.63      0.34      0.44        77
          95       0.62      0.10      0.17        52

    accuracy                           0.62      2466
   macro avg       0.54      0.38      0.40      2466
weighted avg       0.58      0.62      0.58      2466



# Model : Baseline 2 (MultiRocket)

In [17]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(df_y_train["target"])

"""Train MultiROCKET. X must be (n, C, T) — channels-first."""
clf = MultiRocketClassifier(n_kernels=10_000, random_state=SEED)
clf.fit(X_train_cf, y_train_encoded)

y_pred_rocket = clf.predict(X_test_cf)

In [19]:
print("MultiROCKET:")
print(classification_report(
    df_y_test["target"],  
    le.inverse_transform(y_pred_rocket)
))

MultiROCKET:
              precision    recall  f1-score   support

          15       0.63      0.45      0.53       124
          16       0.93      0.93      0.93       270
          42       0.43      0.43      0.43       382
          52       0.00      0.00      0.00        63
          53       1.00      0.57      0.73         7
           6       0.57      0.11      0.19        35
          62       0.25      0.15      0.19       153
          64       0.00      0.00      0.00        24
          65       0.72      0.85      0.78       313
          67       0.21      0.04      0.07        68
          88       0.94      0.84      0.89       121
          90       0.60      0.81      0.69       777
          92       0.92      0.88      0.90        77
          95       0.50      0.08      0.13        52

    accuracy                           0.64      2466
   macro avg       0.55      0.44      0.46      2466
weighted avg       0.60      0.64      0.60      2466

